# 02 · Storage: raw contracts → DuckDB

Loads `data/raw/contracts_<slug>.json` (written by `01_collection.ipynb`) into a small relational schema in a single file, `data/procurement.duckdb`.

| table | one row per | key |
|---|---|---|
| `contracts` | contract (includes the untouched raw record in `raw_record`) | `contract_id` |
| `entities` | company or public body | `nif` |
| `contract_parties` | contract × entity × role (`authority`, `contractor`, `competitor`) | — |
| `contract_cpvs` | contract × CPV code | — |
| `contract_locations` | contract × execution location | — |

Entities are keyed by **NIF**, not by name. The same body is often spelled several ways, and those would otherwise count as different parties.

Loading is an **upsert** on `contract_id`, so re-running it, or loading overlapping batches, never creates duplicates.

**National version.** List the batches written by 01 in `SLUGS` (oldest first) and run all. Each batch is saved to Drive right after it loads, and `load_log` records what was loaded.
The parser accepts both IMPIC formats: recent files pack values into text (`"NIF - Name"`, `dd/mm/yyyy`), older files use objects
(`{"nifEntContrato": …, "descEntContrato": …}`) and ISO dates, and some fields are missing.

**Runs in Google Colab or locally.** The first code cell detects the environment. In Colab it mounts Google Drive and keeps everything under `My Drive/procurement-poc/data/`, so results survive the session.

In [1]:
# ── Environment: Google Colab or local (VS Code / Jupyter) ─────────────────────────────
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_FOLDER = "procurement-poc"   # folder in "My Drive" that holds the project data
PIP_PACKAGES = ["duckdb>=1.1"]

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "data"   # persistent: survives the session
    WORK = Path("/content/work")                                     # fast local disk: wiped when the session ends
    if PIP_PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)
else:
    DATA = Path("data")
    WORK = DATA

DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print(f"running in Colab: {IN_COLAB} | data folder: {DATA}")


# DuckDB is slow and can corrupt on a network drive, so in Colab the notebook works on a local copy
# of the database and copies it back to Drive with push_db(). Locally both paths are the same file.
DB_NAME = "procurement.duckdb"
DB_FILE = WORK / DB_NAME


def pull_db():
    """Colab: copy the database from Drive to local disk before connecting."""
    src = DATA / DB_NAME
    if WORK != DATA and src.exists():
        shutil.copy2(src, DB_FILE)
        print(f"database copied from Drive ({src.stat().st_size / 1e6:.1f} MB)")


def push_db(con=None, sync=True):
    """Colab: save the local copy back to Drive. With an open connection, it checkpoints first.
    sync=True waits until Google has really received the file (unmount + mount), so a crash
    right after this call cannot lose it. Large files take a few minutes."""
    if WORK == DATA:
        return
    if con is not None:
        con.execute("CHECKPOINT")
    tmp = DATA / (DB_NAME + ".tmp")
    shutil.copy2(DB_FILE, tmp)
    tmp.replace(DATA / DB_NAME)
    if sync:
        drive.flush_and_unmount()              # blocks until every pending upload is finished
        drive.mount("/content/drive")
    print(f"database saved to Drive ({DB_FILE.stat().st_size / 1e6:.1f} MB){' and synced' if sync else ''}")

Mounted at /content/drive
running in Colab: True | data folder: /content/drive/MyDrive/procurement-poc/data


In [2]:
import gc
import html
import json
import re

import duckdb
import pandas as pd

# ── Batches to load, OLDEST FIRST: the slugs used in 01 (data/raw/contracts_<slug>.json) ──
# Each batch is saved to Drive as soon as it is loaded, so an interrupted run keeps the finished batches.
# Re-loading a batch never creates duplicates: its contracts are replaced (upsert on contract_id).
SLUGS = [
#    "portugal_2012_2016",
#    "portugal_2017_2019",
#    "portugal_2020",
#    "portugal_2021",
#    "portugal_2022",
#    "portugal_2023",
#    "portugal_2024",
#    "portugal_2025",
    "portugal_2026",
]
SKIP_ALREADY_LOADED = False   # False for the one-time reload of all years (hidden-NIF fix); back to True afterwards.
# skip batches already in load_log with the same file size (resumes a stopped run)
STORE_RAW_RECORD = False      # the full IMPIC JSON stays in data/raw/; storing it again doubles the database

missing = [s for s in SLUGS if not (DATA / "raw" / f"contracts_{s}.json").exists()]
if missing:
    raise FileNotFoundError(f"no raw file for: {missing}  (check the slug names in data/raw/)")
for s in SLUGS:
    f = DATA / "raw" / f"contracts_{s}.json"
    print(f"{f.name:<42} {f.stat().st_size / 1e6:>8.1f} MB")

contracts_portugal_2026.json                  626.6 MB


## 0. Backup
Run once before the full reload. Delete the backup from Drive when the checks in section 3b pass.

In [3]:
# ── One-time backup before the reload (keeps the current database on Drive) ──────────────
import datetime
backup = DATA / f"procurement_backup_{datetime.date.today():%Y%m%d}.duckdb"
if not backup.exists():
    shutil.copy2(DATA / DB_NAME, backup)
print(f"backup: {backup.name} ({backup.stat().st_size / 1e6:,.0f} MB)")


backup: procurement_backup_20261117.duckdb (850 MB)


## 1. Schema

In [4]:
SCHEMA = """
CREATE TABLE IF NOT EXISTS contracts (
    contract_id                BIGINT PRIMARY KEY,
    procedure_id               BIGINT,
    procedure_type             VARCHAR,
    contract_types             VARCHAR,
    object                     VARCHAR,
    description                VARCHAR,
    announcement_number        VARCHAR,
    publication_date           DATE,
    award_decision_date        DATE,
    signing_date               DATE,
    close_date                 DATE,
    base_price                 DECIMAL(18, 2),
    contract_price             DECIMAL(18, 2),
    effective_price            DECIMAL(18, 2),
    execution_days             INTEGER,
    legal_basis                VARCHAR,
    direct_award_justification VARCHAR,
    regime                     VARCHAR,
    award_criteria             VARCHAR,
    framework_agreement        VARCHAR,
    centralized                BOOLEAN,
    green_contract             BOOLEAN,
    documents_link             VARCHAR,
    notes                      VARCHAR,
    year                       INTEGER,
    raw_record                 VARCHAR,
    loaded_at                  TIMESTAMP
);
CREATE TABLE IF NOT EXISTS entities (
    nif  VARCHAR PRIMARY KEY,
    name VARCHAR
);
CREATE TABLE IF NOT EXISTS contract_parties (
    contract_id BIGINT,
    nif         VARCHAR,
    role        VARCHAR,   -- authority | contractor | competitor
    is_sme      BOOLEAN    -- set for contractors only
);
CREATE TABLE IF NOT EXISTS contract_cpvs (
    contract_id     BIGINT,
    cpv_code        VARCHAR,
    cpv_description VARCHAR
);
CREATE TABLE IF NOT EXISTS contract_locations (
    contract_id  BIGINT,
    country      VARCHAR,
    district     VARCHAR,
    municipality VARCHAR
);
-- one row per contract; joint buyers / consortium members are joined with ' + '
CREATE OR REPLACE VIEW v_contracts AS
WITH names AS (
    SELECT p.contract_id, p.role,
           string_agg(p.nif, ' + ' ORDER BY p.nif)   AS nifs,
           string_agg(e.name, ' + ' ORDER BY p.nif)  AS names
    FROM contract_parties p JOIN entities e USING (nif)
    WHERE p.role IN ('authority', 'contractor')
    GROUP BY ALL
)
SELECT c.contract_id, c.publication_date, c.procedure_type, c.object, c.contract_price,
       a.nifs AS authority_nif,  a.names AS authority,
       k.nifs AS contractor_nif, k.names AS contractor
FROM contracts c
LEFT JOIN names a ON a.contract_id = c.contract_id AND a.role = 'authority'
LEFT JOIN names k ON k.contract_id = c.contract_id AND k.role = 'contractor';
-- one row per batch load: which raw file, when, how many contracts
CREATE TABLE IF NOT EXISTS load_log (
    slug       VARCHAR,
    file_bytes BIGINT,
    contracts  INTEGER,
    loaded_at  TIMESTAMP
);
"""

pull_db()
con = duckdb.connect(str(DB_FILE))
con.execute("SET memory_limit = '6GB'")                 # leave room for Python and the Drive upload
con.execute("SET preserve_insertion_order = false")
con.execute(f"SET temp_directory = '{WORK / 'duckdb_tmp'}'")
con.execute(SCHEMA)

database copied from Drive (815.8 MB)


## 2. Parse the raw records into the table shapes
IMPIC changed the format over the years, so each helper accepts both:

| field | recent files (text) | older files (objects) |
|---|---|---|
| parties | `"506361659 - Município da Covilhã"` (bidders double-HTML-escaped) | `{"nifEntContrato": …, "descEntContrato": …, "adjudicatarioPME": …}` |
| CPVs | `"45233140-2 - Obras de estradas"` | object with a code and a description key |
| locations | `"Portugal, Castelo Branco, Covilhã"` | object with country / district / municipality keys |
| dates | `dd/mm/yyyy` | `yyyy-mm-dd` |

Yes/no flags are `"Sim"` / `"Não"`. The quality check after parsing shows whether any value could not be read.

In [5]:
PARTY_RE = re.compile(r"^\s*([^\s-]+)\s*-\s*(.+?)\s*$")
CPV_RE = re.compile(r"^\s*(\d{8}-\d)\s*-\s*(.*?)\s*$")


def _pick(d: dict, *words):
    """First non-empty value whose key contains one of the words (case-insensitive)."""
    for k, v in d.items():
        if any(w in k.lower() for w in words) and v not in (None, ""):
            return str(v).strip()
    return None


def _clean_name(text):
    """Unescape twice (bidders are double-HTML-escaped) and drop the dashes IMPIC leaves in front of hidden NIFs."""
    return html.unescape(html.unescape(str(text or ""))).strip().lstrip("- ").strip()


def _is_nif(nif):
    """IMPIC hides individuals' NIF as '-' (sometimes '--' or ''): those are not NIFs."""
    return bool(nif) and bool(str(nif).strip().strip("-").strip())


def split_party(item):
    """'NIF - Name' (recent) or {'nifEntContrato': ..., 'descEntContrato': ...} (older) -> (nif, name).
    Entries without a real NIF (individuals: '-') get a name-based key, 'NO-NIF:<name>', so different
    people are not merged into one entity and nobody is dropped."""
    if isinstance(item, dict):
        nif = item.get("nifEntContrato") or _pick(item, "nif")
        name = _clean_name(item.get("descEntContrato") or _pick(item, "nome", "name", "design", "denom"))
        if _is_nif(nif):
            nif = str(nif).strip()
            return nif, (name or nif)
        return f"NO-NIF:{name}", name
    text = html.unescape(html.unescape(str(item)))
    m = PARTY_RE.match(text)
    if m and _is_nif(m.group(1)):
        return m.group(1), _clean_name(m.group(2))
    name = _clean_name(text)
    return f"NO-NIF:{name}", name


def sme_flag(value):
    """adjudicatarioPME inside the old party objects: True/False, 'Sim'/'Não', 'S'/'N' or None."""
    if isinstance(value, bool) or value is None:
        return value
    return {"sim": True, "s": True, "true": True, "1": True,
            "não": False, "nao": False, "n": False, "false": False, "0": False}.get(str(value).strip().lower())


def split_cpv(item):
    """'45233140-2 - Obras de estradas' (recent) or an object (older) -> (code, description)."""
    if isinstance(item, dict):
        return _pick(item, "cod", "cpv"), _pick(item, "desc", "design", "nome")
    m = CPV_RE.match(str(item))
    return (m.group(1), m.group(2)) if m else (None, str(item).strip())


def split_location(item):
    """'Portugal, Castelo Branco, Covilhã' (recent) or an object (older) -> (country, district, municipality)."""
    if isinstance(item, dict):
        return (_pick(item, "pais", "país", "country"),
                _pick(item, "distrito", "district"),
                _pick(item, "concelho", "municip"))
    parts = [p.strip() or None for p in str(item).split(",", 2)]
    return tuple((parts + [None, None])[:3])


def to_date(text):
    """Dates come as dd/mm/yyyy (recent files) or yyyy-mm-dd (older files)."""
    if not text:
        return None
    d = pd.to_datetime(text, format="%d/%m/%Y", errors="coerce")
    if pd.isna(d):
        d = pd.to_datetime(text, format="ISO8601", errors="coerce")
    return None if pd.isna(d) else d


def to_number(value):
    """Prices: numbers as published; text like '1234.5' or '1.234,50' is converted; anything else -> None."""
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return value
    s = str(value).strip().replace(" ", "").replace("€", "")
    if "," in s:
        s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def to_int(value):
    n = to_number(value)
    return int(n) if n is not None else None


def yes(text):
    return {"Sim": True, "Não": False}.get(text)


def empty_to_none(text):
    return text.strip() if isinstance(text, str) and text.strip() else None

In [6]:
def parse(records):
    contracts, parties, cpvs, locations = [], [], [], []
    loaded_at = pd.Timestamp.now()

    for r in records:
        g = r.get                      # older IMPIC files do not have every field: missing -> None
        cid = int(r["idcontrato"])
        pub = to_date(g("dataPublicacao"))
        contracts.append({
            "contract_id": cid,
            "procedure_id": to_int(g("idprocedimento")),
            "procedure_type": g("tipoprocedimento"),
            "contract_types": "; ".join(t for t in (g("tipoContrato") or []) if isinstance(t, str)),
            "object": empty_to_none(g("objectoContrato")),
            "description": empty_to_none(g("descContrato")),
            "announcement_number": empty_to_none(g("nAnuncio")),
            "publication_date": pub,
            "award_decision_date": to_date(g("dataDecisaoAdjudicacao")),
            "signing_date": to_date(g("dataCelebracaoContrato")),
            "close_date": to_date(g("dataFechoContrato")),
            "base_price": to_number(g("precoBaseProcedimento")),
            "contract_price": to_number(g("precoContratual")),
            "effective_price": to_number(g("PrecoTotalEfetivo")),
            "execution_days": to_int(g("prazoExecucao")),
            "legal_basis": empty_to_none(g("fundamentacao")),
            "direct_award_justification": empty_to_none(g("fundamentAjusteDireto")),
            "regime": empty_to_none(g("regime")),
            "award_criteria": empty_to_none(g("TipoCriterioAdjudicacao")),
            "framework_agreement": empty_to_none(g("numAcordoQuadro")),
            "centralized": yes(g("ProcedimentoCentralizado")),
            "green_contract": yes(g("ContratEcologico")),
            "documents_link": empty_to_none(g("linkPecasProc")),
            "notes": empty_to_none(g("Observacoes")),
            "year": to_int(g("Ano")) or (pub.year if pub is not None else None),
            "raw_record": json.dumps(r, ensure_ascii=False) if STORE_RAW_RECORD else None,
            "loaded_at": loaded_at,
        })

        # SME flag: recent files list SME NIFs in adjudicatarioPMEs; older files put adjudicatarioPME in each party
        smes = set(r["adjudicatarioPMEs"] or []) if "adjudicatarioPMEs" in r else None
        for role, field in [("authority", "adjudicante"), ("contractor", "adjudicatarios"), ("competitor", "concorrentes")]:
            for item in g(field) or []:
                if item is None:
                    continue
                nif, name = split_party(item)
                if role != "contractor":
                    is_sme = None
                elif isinstance(item, dict):
                    is_sme = sme_flag(item.get("adjudicatarioPME"))
                else:
                    is_sme = (nif in smes) if smes is not None else None
                parties.append({"contract_id": cid, "nif": nif, "name": name, "role": role, "is_sme": is_sme})

        # CPV: recent files use 'cpv' (text); national files use 'CPV' (objects with cpvID / cpvdescricao)
        cpv_items = [x for x in (g("cpv") or []) if x is not None] or [x for x in (g("CPV") or []) if x is not None]
        for item in cpv_items:
            if isinstance(item, dict):
                code = item.get("cpvID") or _pick(item, "cpv", "cod")
                desc = item.get("cpvdescricao") or _pick(item, "desc")
            else:
                code, desc = split_cpv(item)
            cpvs.append({"contract_id": cid, "cpv_code": code, "cpv_description": desc})

        for item in g("localExecucao") or []:
            if item is None:
                continue
            country, district, municipality = split_location(item)
            locations.append({"contract_id": cid, "country": country, "district": district, "municipality": municipality})

    parties = pd.DataFrame(parties, columns=["contract_id", "nif", "name", "role", "is_sme"])
    parties = parties.drop_duplicates(["contract_id", "nif", "role"])
    entities = parties[["nif", "name"]].drop_duplicates("nif", keep="last")  # the latest spelling wins
    return {
        "contracts": pd.DataFrame(contracts),
        "entities": entities,
        "contract_parties": parties.drop(columns="name"),
        "contract_cpvs": pd.DataFrame(cpvs, columns=["contract_id", "cpv_code", "cpv_description"]).drop_duplicates(),
        "contract_locations": pd.DataFrame(locations, columns=["contract_id", "country", "district", "municipality"]).drop_duplicates(),
    }

In [7]:
def quality(tables, raw):
    """Values the parser could not read. Each line should be 0 or very small."""
    c, p, v, l = (tables[k] for k in ["contracts", "contract_parties", "contract_cpvs", "contract_locations"])
    return {
        "no publication date": int(c["publication_date"].isna().sum()),
        "no contract price": int(c["contract_price"].isna().sum()),
        "parties without NIF": int(p["nif"].str.startswith("NO-NIF:").sum()),
        "contracts without CPV": int(len(c) - v["contract_id"].nunique()),
        "contracts without location": int(len(c) - l["contract_id"].nunique()),
        "locations without municipality": int(l["municipality"].isna().sum()),
    }

## 3. Load every batch (upsert)
For each batch, in one transaction:
1. entities: `INSERT OR REPLACE` on `nif`, so a new spelling of a name overwrites the old one
2. every contract in the batch: remove its old rows from `contracts` and the child tables, then insert the fresh ones

Then the database is saved to Drive before the next batch starts. Running a batch twice leaves the tables exactly as they were.

In [8]:
CHILD_TABLES = ["contract_parties", "contract_cpvs", "contract_locations"]


def upsert(con, tables):
    con.execute("BEGIN TRANSACTION")
    try:
        for name, df in tables.items():
            con.register(f"batch_{name}", df)

        con.execute("INSERT OR REPLACE INTO entities BY NAME SELECT * FROM batch_entities")
        for name in ["contracts", *CHILD_TABLES]:
            con.execute(f"DELETE FROM {name} WHERE contract_id IN (SELECT contract_id FROM batch_contracts)")
            con.execute(f"INSERT INTO {name} BY NAME SELECT * FROM batch_{name}")
        con.execute("COMMIT")
    except Exception:
        con.execute("ROLLBACK")
        raise
    finally:
        for name in tables:
            con.unregister(f"batch_{name}")


def row_counts(con):
    names = ["contracts", "entities", *CHILD_TABLES]
    return {n: con.sql(f"SELECT COUNT(*) FROM {n}").fetchall()[0][0] for n in names}


def already_loaded(con, slug, size):
    return con.execute("SELECT COUNT(*) FROM load_log WHERE slug = ? AND file_bytes = ?", [slug, size]).fetchone()[0] > 0


for slug in SLUGS:
    raw_file = DATA / "raw" / f"contracts_{slug}.json"
    size = raw_file.stat().st_size
    if SKIP_ALREADY_LOADED and already_loaded(con, slug, size):
        print(f"── {slug}: already loaded, skipped")
        continue

    print(f"── {slug}: reading {size / 1e6:.0f} MB ...")
    raw = json.loads(raw_file.read_text(encoding="utf-8"))
    tables = parse(raw)
    q = quality(tables, raw)
    del raw
    gc.collect()

    upsert(con, tables)
    con.execute("INSERT INTO load_log VALUES (?, ?, ?, ?)", [slug, size, len(tables["contracts"]), pd.Timestamp.now()])
    print(f"   loaded {len(tables['contracts']):,} contracts | quality: {q}")
    del tables
    gc.collect()

    push_db(con)            # save to Drive after every batch
    print(f"   totals now: {row_counts(con)}")

print("\nall batches done")

── portugal_2026: reading 627 MB ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   loaded 212,076 contracts | quality: {'no publication date': 0, 'no contract price': 0, 'parties without NIF': 53316, 'contracts without CPV': 43, 'contracts without location': 242, 'locations without municipality': 29572}
Mounted at /content/drive
database saved to Drive (921.7 MB) and synced
   totals now: {'contracts': 2288545, 'entities': 236247, 'contract_parties': 8083643, 'contract_cpvs': 2328093, 'contract_locations': 2434775}

all batches done


In [9]:
# Load history: one line per batch (a batch loaded twice is fine — the upsert replaced its rows)
con.sql("SELECT slug, contracts, ROUND(file_bytes / 1e6) AS file_mb, loaded_at FROM load_log ORDER BY loaded_at").df()

,slug,contracts,file_mb,loaded_at
0,portugal_2012_2016,469901,1206.0,2026-11-16 04:39:14.574639
1,portugal_2017_2019,401192,1066.0,2026-11-16 04:51:32.483988
2,portugal_2020,176410,455.0,2026-11-16 04:59:23.304689
3,portugal_2021,187267,512.0,2026-11-16 05:07:50.531802
4,portugal_2022,176304,497.0,2026-11-16 05:16:06.955991
5,portugal_2023,191883,550.0,2026-11-16 05:24:32.634825
6,portugal_2024,226174,647.0,2026-11-16 12:07:37.513269
7,portugal_2025,247338,730.0,2026-11-16 12:22:51.730615
8,portugal_2026,212076,627.0,2026-11-16 12:38:28.668652
9,portugal_2012_2016,469901,1206.0,2026-11-17 03:19:00.466430


## 3b. Clean up after the hidden-NIF fix
The reload replaced every contract's parties, so the old keys (`'-'`, `'NO-NIF:--…'`) have no links left. Remove those orphan entities and check the result.

In [10]:
orphans = con.sql("SELECT COUNT(*) FROM entities WHERE nif NOT IN (SELECT DISTINCT nif FROM contract_parties)").fetchone()[0]
con.execute("DELETE FROM entities WHERE nif NOT IN (SELECT DISTINCT nif FROM contract_parties)")
print(f"orphan entities removed: {orphans:,}")

print(con.sql("""
    SELECT
      (SELECT COUNT(*) FROM contract_parties WHERE nif IN ('-', '--', ''))   AS parties_with_dash_nif,   -- must be 0
      (SELECT COUNT(*) FROM entities WHERE nif LIKE 'NO-NIF:-%')            AS names_starting_with_dash, -- must be 0
      (SELECT COUNT(*) FROM entities WHERE nif LIKE 'NO-NIF:%')             AS entities_without_nif
""").df())
push_db(con)


orphan entities removed: 11,681
   parties_with_dash_nif  names_starting_with_dash  entities_without_nif
0                      0                         0                114138
Mounted at /content/drive
database saved to Drive (921.4 MB) and synced


In [11]:
# Test case from the news (Página Um, 22/08/2024): Município da Covilhã and Jorge Gaspar, 2014–2024
con.sql("""
    SELECT contract_id, publication_date, contract_price, contractor
    FROM v_contracts
    WHERE authority_nif = '505330768' AND contractor ILIKE '%gaspar%'
    ORDER BY publication_date
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,contract_id,publication_date,contract_price,contractor
0,1081284,2014-05-29,48000.0,"Antunes Ferreira, Jorge Gaspar & Associados, RL"
1,2293635,2016-07-15,48000.0,Jorge Baptista Carpinteiro Gaspar
2,3514106,2017-07-13,48000.0,Jorge Baptista Carpinteiro Gaspar
3,7752176,2021-07-09,144000.0,Jorge Baptista Carpinteiro Gaspar
4,8423631,2021-11-25,48000.0,Jorge Baptista Carpinteiro Gaspar
5,10851781,2024-08-02,144000.0,Jorge Baptista Carpinteiro Gaspar


## 4. Sanity checks
A few queries against the stored data. They also show the schema can answer the questions the project needs: concentration between the same authority and contractor, the procedure mix, and single bidding.

In [12]:
#con.sql("""
#    SELECT MIN(publication_date) AS first_published, MAX(publication_date) AS last_published,
#           COUNT(*) AS contracts, SUM(contract_price) AS total_value,
#           COUNT(*) FILTER (WHERE contract_price IS NULL) AS missing_price
#    FROM contracts
#""").df()

In [13]:
# Per year: contracts, missing dates, CPV and location coverage
#con.sql("""
#    SELECT c.year,
#           COUNT(*)                                                                   AS contracts,
#           COUNT(*) FILTER (c.publication_date IS NULL)                              AS no_pub_date,
#           COUNT(*) FILTER (c.contract_id NOT IN (SELECT contract_id FROM contract_cpvs))      AS no_cpv,
#           COUNT(*) FILTER (c.contract_id NOT IN (SELECT contract_id FROM contract_locations)) AS no_location,
#           ROUND(SUM(c.contract_price) / 1e6, 1)                                     AS value_meur
#    FROM contracts c GROUP BY c.year ORDER BY c.year
#""").df()

In [14]:
# No duplicates across batches: rows must equal distinct ids
#con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT contract_id) AS ids FROM contracts").df()

In [15]:
# Which authority -> contractor pairs account for the most money? (concentration / favouritism signal)
#con.sql("""
#    SELECT authority, contractor, COUNT(*) AS contracts, SUM(contract_price) AS total_value
#    FROM v_contracts
#    GROUP BY ALL
#    ORDER BY total_value DESC
#    LIMIT 10
#""").df()

In [16]:
# Procedure mix, and how many procedures received only one bid
# (a single bid is expected for direct awards; it's a red flag only for competitive procedures like Concurso público)
#con.sql("""
#    WITH bids AS (
#        SELECT contract_id, COUNT(*) AS bidders
#        FROM contract_parties WHERE role = 'competitor' GROUP BY contract_id
#   )
#    SELECT c.procedure_type,
#           COUNT(*)                                   AS contracts,
#           SUM(c.contract_price)                      AS total_value,
#           ROUND(AVG(b.bidders), 1)                   AS avg_bidders,
#           COUNT(*) FILTER (WHERE b.bidders = 1)      AS single_bid
#    FROM contracts c LEFT JOIN bids b USING (contract_id)
#    GROUP BY ALL
#    ORDER BY contracts DESC
#""").df()

In [17]:
# One entity, many spellings: needs raw_record, which is not stored in the national version
#if STORE_RAW_RECORD:
#    display(con.sql("""
#        SELECT p.nif, COUNT(DISTINCT json_extract_string(c.raw_record, '$.adjudicante[0]')) AS spellings_in_raw
#        FROM contract_parties p JOIN contracts c USING (contract_id)
#        WHERE p.role = 'authority'
#        GROUP BY p.nif HAVING spellings_in_raw > 1
#        ORDER BY spellings_in_raw DESC
#    """).df().merge(con.sql("SELECT * FROM entities").df(), on="nif"))
#else:
#    print("skipped: raw_record is not stored (STORE_RAW_RECORD = False)")

## 5. Compact the database (run once, after all batches)
Replacing rows leaves unused space inside the file, and rows loaded before `STORE_RAW_RECORD = False` still carry their JSON.
This cell clears the old `raw_record` values, writes a fresh copy of every table (documents and RAG included) and swaps it in.

In [18]:
RUN_COMPACT = True

if RUN_COMPACT:
    if not STORE_RAW_RECORD:
        con.execute("UPDATE contracts SET raw_record = NULL WHERE raw_record IS NOT NULL")
    con.execute("CHECKPOINT")
    before = DB_FILE.stat().st_size
    compact_file = WORK / "procurement_compact.duckdb"
    compact_file.unlink(missing_ok=True)
    con.execute(f"ATTACH '{compact_file}' AS compact")
    con.execute("COPY FROM DATABASE procurement TO compact")
    con.execute("DETACH compact")
    con.close()
    compact_file.replace(DB_FILE)
    con = duckdb.connect(str(DB_FILE))
    print(f"compacted: {before / 1e6:,.0f} MB -> {DB_FILE.stat().st_size / 1e6:,.0f} MB")
    print(row_counts(con))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

compacted: 921 MB -> 817 MB
{'contracts': 2288545, 'entities': 224566, 'contract_parties': 8083643, 'contract_cpvs': 2328093, 'contract_locations': 2434775}


In [19]:
con.close()
push_db()   # Colab: copy the database back to Drive and wait until the upload is finished

Mounted at /content/drive
database saved to Drive (816.9 MB) and synced
